# Module 1 — Classification Training (DRG-Net, ResNet50)

Runs DRG-Net's own `dr_classification/main.py` unmodified, with our config
(`module1/configs/fgadr_poc.yaml`) pointed at our local FGADR copy and Drive save paths.
**Epochs (20 for FGADR, 60 for IDRiD) are kept exactly as published** -- this run is meant to
match the literature, not a reduced POC (unlike the segmentation half in notebook 03).

Run notebook 01 first (same Colab runtime, or re-run its cells if this is a fresh session).


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import getpass, os
# This repo is PRIVATE -- Colab needs a GitHub Personal Access Token to clone it.
# Create one (once, reusable across all 4 notebooks/sessions) at
# https://github.com/settings/tokens -> "Generate new token (classic)" -> scope: repo.
# Input is hidden; not saved anywhere.
GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token (repo scope): ')

DRIVE_DATA_DIR = '/content/drive/MyDrive/Thesis_Datasets'  # <-- change if you used a different folder
assert os.path.isdir(DRIVE_DATA_DIR), f"Not found: {DRIVE_DATA_DIR} -- upload the 4 dataset zips there first"


Mounted at /content/drive
GitHub Personal Access Token (repo scope): ··········


In [2]:
# Unzip datasets locally on the Colab VM disk (much faster I/O than reading zips off Drive
# directly). -n skips files that already exist, so this is safe/cheap to re-run.
os.makedirs('/content/data', exist_ok=True)
%cd /content/data
!unzip -q -n "$DRIVE_DATA_DIR/FGADR-Seg-set_Release.zip"
!unzip -q -n "$DRIVE_DATA_DIR/archive.zip" -d IDRiD_dataset_root
!unzip -q -n "$DRIVE_DATA_DIR/FIRE_dataset.zip"
!unzip -q -n "$DRIVE_DATA_DIR/LongDRScreening_20150209.zip"

# archive.zip may unzip with an extra nesting level; normalize so IDRiD_dataset ends up
# directly under /content/data
import glob, shutil
candidates = glob.glob('/content/data/IDRiD_dataset_root/**/IDRiD_dataset', recursive=True)
if candidates and not os.path.isdir('/content/data/IDRiD_dataset'):
    shutil.move(candidates[0], '/content/data/IDRiD_dataset')
print('IDRiD_dataset present:', os.path.isdir('/content/data/IDRiD_dataset'))


/content/data
IDRiD_dataset present: True


In [3]:
# Clone this repo (private -- uses the token above) and the pinned DRG-Net reference
# implementation. Skips cleanly if already cloned in this runtime.
%cd /content
if not os.path.isdir('/content/M2-DRProgression'):
    !git clone --branch module1-fgadr-poc https://{GITHUB_TOKEN}@github.com/bearawr/M2-DRProgression.git M2-DRProgression
if not os.path.isdir('/content/dr-joint-learning'):
    !git clone https://github.com/DFKI-Interactive-Machine-Learning/dr-joint-learning.git dr-joint-learning
    %cd dr-joint-learning
    !git checkout 0da1bbe885f438390a0e94ec486283d9b21d5547
    %cd /content


/content
Cloning into 'M2-DRProgression'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 141 (delta 66), reused 122 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 85.33 KiB | 1.29 MiB/s, done.
Resolving deltas: 100% (66/66), done.
Cloning into 'dr-joint-learning'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (283/283), done.
remote: Compressing objects: 100% (187/187), done.
remote: Total 283 (delta 110), reused 249 (delta 88), pack-reused 0 (from 0)
Receiving objects: 100% (283/283), 9.32 MiB | 13.69 MiB/s, done.
Resolving deltas: 100% (110/110), done.
/content/dr-joint-learning
Note: switching to '0da1bbe885f438390a0e94ec486283d9b21d5547'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switc

In [4]:
# Copy DRG-Net's own filtered-label CSV (vendored in our repo) into the local FGADR folder --
# dr_segmentation/utils.py::get_images_fgadr_from_pd reads this file from inside IMAGE_DIR.
# Made robust to zip-nesting variation (some zip tools add/drop the top-level
# FGADR-Seg-set_Release wrapper folder) by locating Seg-set/ wherever it actually landed.
import glob as _glob, os, shutil

_candidates = _glob.glob('/content/data/**/Seg-set', recursive=True)
assert _candidates, (
    'No Seg-set folder found under /content/data. Run: !ls /content/data  and  '
    '!ls "$DRIVE_DATA_DIR"  to check the zip is named exactly FGADR-Seg-set_Release.zip '
    'and actually unzipped in the previous cell.'
)
fgadr_segset_dir = _candidates[0]
print('Found FGADR Seg-set at:', fgadr_segset_dir)

expected = '/content/data/FGADR-Seg-set_Release/Seg-set'
if fgadr_segset_dir != expected:
    os.makedirs(os.path.dirname(expected), exist_ok=True)
    if not os.path.exists(expected):
        os.symlink(fgadr_segset_dir, expected)
    print(f'Normalized path: {expected} -> {fgadr_segset_dir}')

shutil.copy(
    '/content/M2-DRProgression/module1/data/DR_Seg_Grading_Label_Filtered.csv',
    os.path.join(expected, 'DR_Seg_Grading_Label_Filtered.csv')
)
print('done')


Found FGADR Seg-set at: /content/data/Seg-set
Normalized path: /content/data/FGADR-Seg-set_Release/Seg-set -> /content/data/Seg-set
done


In [5]:
%cd /content/dr-joint-learning/dr_classification
# NOT installing the repo's full requirements.txt -- it pins torch==1.7.1, numpy==1.21.6,
# Pillow==6.2.2, etc. (2022-era versions). Colab already ships modern, GPU-matched
# torch/numpy/pandas/opencv/Pillow/scikit-learn -- reinstalling those old pins would try to
# BUILD them from source against Colab's much newer Python (they have no prebuilt wheels for
# it), which is exactly the "Getting requirements to build wheel" error, and even if it
# somehow succeeded it would downgrade/break Colab's working CUDA-enabled torch. Instead,
# install only the packages Colab does NOT already have that this repo's imports need.
!pip install -q munch self-attention-cv pretrainedmodels cnn_finetune


/content/dr-joint-learning/dr_classification
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.8 MB/s eta 0:00:00


In [6]:
# Copy our POC configs in next to DRG-Net's own
!cp /content/M2-DRProgression/module1/configs/fgadr_poc.yaml configs/fgadr_poc.yaml
!cp /content/M2-DRProgression/module1/configs/idrid_poc.yaml configs/idrid_poc.yaml
!mkdir -p /content/drive/MyDrive/Thesis_Datasets/module1_runs/classification/fgadr/saves
!mkdir -p /content/drive/MyDrive/Thesis_Datasets/module1_runs/classification/fgadr/logs
!mkdir -p /content/drive/MyDrive/Thesis_Datasets/module1_runs/classification/idrid/saves
!mkdir -p /content/drive/MyDrive/Thesis_Datasets/module1_runs/classification/idrid/logs


In [11]:
!sed -i 's/fillcolor=aug_args.value_fill/fill=aug_args.value_fill/' /content/dr-joint-learning/dr_classification/data/transforms.py


In [12]:
# FGADR grading model -- 20 epochs, as published
!python main.py -c configs/fgadr_poc.yaml -overwrite



2026-09-06 14:55:15.055029: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/content/drive/MyDrive/Thesis_Datasets/module1_runs/classification/fgadr/saves
LOADING CONFIG FILE: configs/fgadr_poc.yaml
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.

In [13]:
# IDRiD grading model -- 60 epochs, as published (skip/interrupt if short on time;
# FGADR is the newly-added dataset and the priority for tomorrow's presentation)
!python main.py -c configs/idrid_poc.yaml -overwrite


2026-09-06 15:30:08.463998: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/content/drive/MyDrive/Thesis_Datasets/module1_runs/classification/idrid/saves
LOADING CONFIG FILE: configs/idrid_poc.yaml
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.

## What to pull into slides (b.5)

`main.py` auto-evaluates the best-validation and final checkpoints on the held-out test set
at the end of each run and prints: accuracy, quadratic-weighted kappa (QWK), and the full
confusion matrix. TensorBoard logs (loss/accuracy/kappa curves) are under
`.../classification/<dataset>/logs` -- `%load_ext tensorboard` +
`%tensorboard --logdir /content/drive/MyDrive/Thesis_Datasets/module1_runs/classification/fgadr/logs`
to view them inline.
